# Exploration de la base Rome

Notebook jetable pour se faire une idée des données. Choisir un fichier dans la cellule **Choix du graphe**
(ou laisser `FICHIER = None` pour en tirer un au hasard), puis tout exécuter.

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

## Choix du graphe

In [ ]:
DATA_DIR = Path("data/rome")

# Nom d'un fichier de data/rome (ex. "grafo2774.26.graphml"), ou None pour un tirage au hasard.
FICHIER = None

fichiers = sorted(DATA_DIR.glob("*.graphml"))
print(f"{len(fichiers)} graphes dans {DATA_DIR}")

chemin = DATA_DIR / FICHIER if FICHIER else random.choice(fichiers)
print("Fichier choisi :", chemin.name)

## Chargement

In [ ]:
G = nx.read_graphml(chemin)
print(G)
print("Orienté :", G.is_directed())
print("Exemples de nœuds :", list(G.nodes)[:5])
print("Exemples d'arêtes :", list(G.edges)[:5])
print("Attributs d'un nœud :", G.nodes[next(iter(G.nodes))])

## Statistiques de base

In [ ]:
degres = np.array([d for _, d in G.degree()])
composantes = list(nx.connected_components(G))

print(f"Nœuds              : {G.number_of_nodes()}")
print(f"Arêtes             : {G.number_of_edges()}")
print(f"Densité            : {nx.density(G):.3f}")
print(f"Composantes        : {len(composantes)}")
print(f"Degré min/moy/max  : {degres.min()} / {degres.mean():.2f} / {degres.max()}")
print(f"Clustering moyen   : {nx.average_clustering(G):.3f}")
if nx.is_connected(G):
    print(f"Diamètre           : {nx.diameter(G)}")
print(f"Planaire           : {nx.check_planarity(G)[0]}")

## Distribution des degrés

In [ ]:
plt.figure(figsize=(5, 3))
plt.hist(degres, bins=range(degres.min(), degres.max() + 2), align="left", rwidth=0.8)
plt.xlabel("degré")
plt.ylabel("nombre de nœuds")
plt.title(chemin.name)
plt.show()

## Deux dessins classiques

- `spring_layout` : modèle de forces (attraction sur les arêtes, répulsion entre nœuds).
- `kamada_kawai_layout` : minimise le **stress**, c'est-à-dire l'écart entre distances dessinées et distances dans le graphe.

In [ ]:
dessins = {
    "spring (forces)": nx.spring_layout(G, seed=0),
    "kamada-kawai (stress)": nx.kamada_kawai_layout(G),
}

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, (nom, pos) in zip(axes, dessins.items()):
    nx.draw(G, pos, ax=ax, node_size=60, width=0.8, with_labels=False)
    ax.set_title(nom)
plt.show()

## Distances dans le graphe et stress

La matrice des plus courts chemins $d_{ij}$ est ce que le stress cherche à respecter :
$\text{stress} = \sum_{i<j} \frac{(\lVert x_i - x_j \rVert - d_{ij})^2}{d_{ij}^2}$.
Le dessin est d'abord remis à la meilleure échelle, sinon le score dépend de l'unité choisie.

In [ ]:
noeuds = list(G.nodes)
D = nx.floyd_warshall_numpy(G, nodelist=noeuds)

plt.figure(figsize=(4.5, 4))
plt.imshow(D, cmap="viridis")
plt.colorbar(label="plus court chemin")
plt.title("Matrice des distances")
plt.show()

def stress(pos, D, noeuds):
    X = np.array([pos[n] for n in noeuds])
    iu = np.triu_indices(len(noeuds), k=1)
    d = D[iu]
    ok = np.isfinite(d)                       # ignore les paires dans des composantes différentes
    e = np.linalg.norm(X[:, None] - X[None, :], axis=-1)[iu][ok]
    d, w = d[ok], 1 / d[ok] ** 2
    s = (w * e * d).sum() / (w * e * e).sum()  # échelle optimale
    return (w * (s * e - d) ** 2).sum()

for nom, pos in dessins.items():
    print(f"{nom:24s} stress = {stress(pos, D, noeuds):.2f}")

## Vue d'ensemble de la base (échantillon)

Lit un échantillon de fichiers pour voir la répartition des tailles, et compare le nombre
après le point dans le nom de fichier au nombre réel de nœuds.

In [ ]:
TAILLE_ECHANTILLON = 300

echantillon = random.sample(fichiers, TAILLE_ECHANTILLON)
lignes = []
for f in echantillon:
    g = nx.read_graphml(f)
    suffixe = int(f.name.split(".")[1])
    lignes.append((f.name, g.number_of_nodes(), g.number_of_edges(), suffixe))

n = np.array([l[1] for l in lignes])
m = np.array([l[2] for l in lignes])
s = np.array([l[3] for l in lignes])

print(f"Nœuds  : min {n.min()}, médiane {int(np.median(n))}, max {n.max()}")
print(f"Arêtes : min {m.min()}, médiane {int(np.median(m))}, max {m.max()}")
print(f"Nombre du nom de fichier = nombre de nœuds : {(s == n).sum()}/{len(lignes)}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(n, bins=30)
axes[0].set_xlabel("nœuds")
axes[1].scatter(n, m, s=8)
axes[1].set_xlabel("nœuds")
axes[1].set_ylabel("arêtes")
plt.show()